# Notebook 01: Input Loading

**Goal:** Accept any document input (digital PDF, scanned PDF, or image file) and convert it into a list of page images saved to `data/page_images/`.

**Supported inputs:**
- Digital PDF (with selectable text)
- Scanned PDF (image-only pages)
- Image files: PNG, JPG, JPEG, TIFF, BMP

**Output:** Page images saved as PNG files in `data/page_images/` + `metadata.json`

In [ ]:
# Install dependencies (run once)
# !pip install pdf2image Pillow matplotlib
# System dependency: sudo apt-get install -y poppler-utils

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Add project root to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    DATA_DIR, PAGE_IMAGES_DIR, SAMPLES_DIR,
    save_json, save_image, display_images,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Samples dir: {SAMPLES_DIR}")

## Configuration

Set the path to your input file below. It can be:
- A PDF file (digital or scanned): `samples/textbook_page.pdf`
- An image file: `samples/textbook_page.png`

In [ ]:
# ── CONFIGURE YOUR INPUT HERE ──────────────────────────────────────────────────
INPUT_FILE = SAMPLES_DIR / "sample.pdf"  # Change this to your file path
DPI = 300  # Resolution for PDF-to-image conversion (300 recommended for OCR)
# ──────────────────────────────────────────────────────────────────────────────

## Input Detection & Loading

The loader detects the input type by file extension, then converts to a list of PIL Images.

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp", ".webp"}
PDF_EXTENSIONS = {".pdf"}


def detect_input_type(file_path: Path) -> str:
    """Detect whether input is a PDF or image file."""
    ext = file_path.suffix.lower()
    if ext in PDF_EXTENSIONS:
        return "pdf"
    elif ext in IMAGE_EXTENSIONS:
        return "image"
    else:
        raise ValueError(f"Unsupported file type: {ext}. Supported: {PDF_EXTENSIONS | IMAGE_EXTENSIONS}")


def load_pdf_as_images(file_path: Path, dpi: int = 300) -> list[Image.Image]:
    """Convert each page of a PDF to a PIL Image."""
    from pdf2image import convert_from_path
    
    print(f"Converting PDF to images at {dpi} DPI...")
    images = convert_from_path(str(file_path), dpi=dpi)
    print(f"  Converted {len(images)} page(s)")
    return images


def load_image_file(file_path: Path) -> list[Image.Image]:
    """Load a single image file as a list with one PIL Image."""
    print(f"Loading image file: {file_path.name}")
    img = Image.open(str(file_path)).convert("RGB")
    print(f"  Image size: {img.size[0]}x{img.size[1]}")
    return [img]


def load_input(file_path: Path, dpi: int = 300) -> tuple[list[Image.Image], str]:
    """
    Load any supported input file and return page images.
    
    Returns:
        (list of PIL Images, input_type string)
    """
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Input file not found: {file_path}")
    
    input_type = detect_input_type(file_path)
    print(f"Input type detected: {input_type}")
    print(f"File: {file_path.name} ({file_path.stat().st_size / 1024:.1f} KB)")
    
    if input_type == "pdf":
        images = load_pdf_as_images(file_path, dpi=dpi)
    else:
        images = load_image_file(file_path)
    
    return images, input_type


print("Input loader ready.")

## Load Input File

Run the loader on the configured input file.

In [ ]:
# Load the input
page_images, input_type = load_input(INPUT_FILE, dpi=DPI)

print(f"\nLoaded {len(page_images)} page(s)")
for i, img in enumerate(page_images):
    print(f"  Page {i}: {img.size[0]}x{img.size[1]} pixels, mode={img.mode}")

## Display Page Images

Visual inspection of loaded pages.

In [ ]:
# Display all loaded pages
titles = [f"Page {i} ({img.size[0]}x{img.size[1]})" for i, img in enumerate(page_images)]
display_images(page_images, titles=titles, cols=min(len(page_images), 3))

## Save Page Images to `data/page_images/`

Save the loaded pages as PNG files along with metadata, so subsequent notebooks can load them.

In [ ]:
# Clear previous outputs
import shutil
if PAGE_IMAGES_DIR.exists():
    shutil.rmtree(PAGE_IMAGES_DIR)
PAGE_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Save each page image
saved_paths = []
for i, img in enumerate(page_images):
    page_path = PAGE_IMAGES_DIR / f"page_{i}.png"
    save_image(img, page_path)
    saved_paths.append(str(page_path))
    print(f"Saved: {page_path.name} ({img.size[0]}x{img.size[1]})")

# Save metadata
metadata = {
    "source_file": str(INPUT_FILE),
    "input_type": input_type,
    "num_pages": len(page_images),
    "dpi": DPI,
    "pages": [
        {
            "index": i,
            "filename": f"page_{i}.png",
            "width": img.size[0],
            "height": img.size[1],
        }
        for i, img in enumerate(page_images)
    ],
}
save_json(metadata, PAGE_IMAGES_DIR / "metadata.json")

print(f"\nSaved {len(page_images)} page(s) to {PAGE_IMAGES_DIR}")
print(f"Metadata saved to {PAGE_IMAGES_DIR / 'metadata.json'}")

## Verify Saved Output

Quick check that everything was saved correctly and can be reloaded.

In [ ]:
# Verify: reload metadata and first page image
from src.utils import load_json, load_image

reloaded_meta = load_json(PAGE_IMAGES_DIR / "metadata.json")
print("Metadata:")
print(json.dumps(reloaded_meta, indent=2))

# Reload and display first page
first_page = load_image(PAGE_IMAGES_DIR / "page_0.png")
print(f"\nReloaded page_0.png: {first_page.size[0]}x{first_page.size[1]}")

plt.figure(figsize=(8, 10))
plt.imshow(np.array(first_page))
plt.title("Reloaded Page 0")
plt.axis("off")
plt.show()

print("\n✓ Output verified. Next notebook: 02_ocr_comparison.ipynb")